# Verilerin API  Çekme (JSOC' dan çekiyoruz) csv ye kaydetme

her 12 dakikada bir tüm verileri çekiyoruz ve en aktif bölgeyi alıyoruz R_VALUE değeri en yüksek olanı en aktif bölge olarak seçiyoruz.

Her 12 dakikada bir:

1-o ana ait tüm aktif bölgeleri çek
<br>2-her bölge için R_VALUE değerine bak
<br>3-R_VALUE’su en yüksek olanı “en aktif bölge” kabul et
<br>4-o bölgenin verisini kullan

In [20]:
import time
from datetime import datetime, timedelta
import os

import drms
import pandas as pd

FEATURES = [
    "R_VALUE", "TOTUSJH", "TOTBSQ", "TOTPOT", "TOTUSJZ", "ABSNJZH",
    "SAVNCPP", "USFLUX", "TOTFZ", "MEANPOT", "EPSX", "EPSY", "EPSZ",
    "MEANSHR", "SHRGT45", "MEANGAM", "MEANGBT", "MEANGBZ", "MEANGBH",
    "MEANJZH", "TOTFY", "MEANJZD", "MEANALP", "TOTFX"
]

KEYS = ["HARPNUM", "T_REC", "NOAA_ARS"] + FEATURES
POLL_SECONDS = 720
CSV_PATH = "most_active_region_log.csv"


def build_recent_query(hours=6):
    start = datetime.utcnow() - timedelta(hours=hours)
    start_str = start.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    return f"hmi.sharp_720s_nrt[][{start_str}/{hours}h@12m]"


def fetch_recent_regions(client, hours=6):
    query = build_recent_query(hours)
    print("Sorgu:", query)

    df = client.query(query, key=",".join(KEYS))
    if df is None or df.empty:
        return pd.DataFrame()
    return df


def choose_latest_timeslot(df):
    temp = df.copy()
    temp["T_REC_DT"] = drms.to_datetime(temp["T_REC"])
    temp = temp.dropna(subset=["T_REC_DT"])

    if temp.empty:
        return pd.DataFrame()

    latest_t = temp["T_REC_DT"].max()
    latest_df = temp[temp["T_REC_DT"] == latest_t].copy()
    return latest_df


def choose_most_active_region(latest_df):
    temp = latest_df.copy()
    temp["R_VALUE"] = pd.to_numeric(temp["R_VALUE"], errors="coerce")
    temp = temp.dropna(subset=["R_VALUE"])

    if temp.empty:
        return None

    temp = temp.sort_values("R_VALUE", ascending=False)
    return temp.iloc[0]


def save_best_row_to_csv(best_row, csv_path):
    row_df = pd.DataFrame([best_row])

    if os.path.exists(csv_path):
        old_df = pd.read_csv(csv_path)
        combined = pd.concat([old_df, row_df], ignore_index=True)

        combined = combined.drop_duplicates(
            subset=["HARPNUM", "T_REC"],
            keep="last"
        )
        combined.to_csv(csv_path, index=False)
    else:
        row_df.to_csv(csv_path, index=False)


def main():
    client = drms.Client()
    last_seen_t_rec = None

    print("İzleme başladı...")

    while True:
        try:
            df = fetch_recent_regions(client, hours=1)

            if df.empty:
                print("1 saat boş, 6 saate geçiliyor")
                df = fetch_recent_regions(client, hours=6)

            if df.empty:
                print("Hiç kayıt bulunamadı.")
            else:
                print("Toplam kayıt:", len(df))

                latest_df = choose_latest_timeslot(df)

                if latest_df.empty:
                    print("Geçerli T_REC bulunamadı.")
                else:
                    print(latest_df[["HARPNUM", "NOAA_ARS", "R_VALUE"]].sort_values("R_VALUE", ascending=False).head(10))
                    best = choose_most_active_region(latest_df)

                    if best is None:
                        print("R_VALUE içeren kayıt bulunamadı.")
                    else:
                        if best["T_REC"] != last_seen_t_rec:
                            last_seen_t_rec = best["T_REC"]
                            print("\nYeni zaman dilimi bulundu:")
                        else:
                            print("\nAynı zaman dilimi tekrar okundu:")

                        print(f"T_REC    : {best['T_REC']}")
                        print(f"HARPNUM  : {best['HARPNUM']}")
                        print(f"NOAA_ARS : {best.get('NOAA_ARS')}")
                        print(f"R_VALUE  : {best.get('R_VALUE')}")

                        save_best_row_to_csv(best, CSV_PATH)
                        print(f"Kayıt dosyaya yazıldı: {CSV_PATH}")

        except Exception as e:
            print("Hata:", e)

        time.sleep(POLL_SECONDS)


if __name__ == "__main__":
    main()

İzleme başladı...
Sorgu: hmi.sharp_720s_nrt[][2026.03.27_21:54:49_TAI/1h@12m]
1 saat boş, 6 saate geçiliyor
Sorgu: hmi.sharp_720s_nrt[][2026.03.27_16:54:51_TAI/6h@12m]
Toplam kayıt: 262
     HARPNUM NOAA_ARS  R_VALUE
77     13371    14400    4.191
54     13370    14398    3.274
123    13376    14399    3.079
22     13363    14397    0.000
100    13372  MISSING    0.000

Yeni zaman dilimi bulundu:
T_REC    : 2026.03.27_21:48:00_TAI
HARPNUM  : 13371
NOAA_ARS : 14400
R_VALUE  : 4.191
Kayıt dosyaya yazıldı: most_active_region_log.csv
Sorgu: hmi.sharp_720s_nrt[][2026.03.27_22:06:54_TAI/1h@12m]
1 saat boş, 6 saate geçiliyor
Sorgu: hmi.sharp_720s_nrt[][2026.03.27_17:06:55_TAI/6h@12m]
Toplam kayıt: 261
    HARPNUM NOAA_ARS  R_VALUE
76    13371    14400    4.264
53    13370    14398    3.312
22    13363    14397    0.000

Yeni zaman dilimi bulundu:
T_REC    : 2026.03.27_22:00:00_TAI
HARPNUM  : 13371
NOAA_ARS : 14400
R_VALUE  : 4.264
Kayıt dosyaya yazıldı: most_active_region_log.csv


KeyboardInterrupt: 

# SQLite da raw_dosyasına kaydetme

In [27]:
import sqlite3
from datetime import datetime, timedelta
import math

import drms
import pandas as pd


# -----------------------------
# AYARLAR
# -----------------------------
DB_PATH = "/Users/mustafaseyyitdogan/Desktop/detection_of_solar_flares/hackathon_detection_of_solar_flares/app/modules/data/fetch_data/raw_data/space_weather.db"
TABLE_NAME = "sharp_raw"

FEATURES = [
    "R_VALUE", "TOTUSJH", "TOTBSQ", "TOTPOT", "TOTUSJZ", "ABSNJZH",
    "SAVNCPP", "USFLUX", "TOTFZ", "MEANPOT", "EPSX", "EPSY", "EPSZ",
    "MEANSHR", "SHRGT45", "MEANGAM", "MEANGBT", "MEANGBZ", "MEANGBH",
    "MEANJZH", "TOTFY", "MEANJZD", "MEANALP", "TOTFX"
]

KEYS = ["HARPNUM", "T_REC", "NOAA_ARS"] + FEATURES


# -----------------------------
# 1) JSOC SORGUSU
# -----------------------------
def build_recent_query(hours=6):
    start = datetime.utcnow() - timedelta(hours=hours)
    start_str = start.strftime("%Y.%m.%d_%H:%M:%S_TAI")
    return f"hmi.sharp_720s_nrt[][{start_str}/{hours}h@12m]"


def fetch_recent_regions(client, hours=6):
    query = build_recent_query(hours)
    print("Sorgu:", query)

    df = client.query(query, key=",".join(KEYS))
    if df is None or df.empty:
        return pd.DataFrame()

    return df


# -----------------------------
# 2) SQLITE TABLO OLUŞTURMA
# -----------------------------
def create_table_if_not_exists(conn):
    cursor = conn.cursor()

    cursor.execute(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
        id INTEGER PRIMARY KEY AUTOINCREMENT,

        harpnum INTEGER NOT NULL,
        t_rec TEXT NOT NULL,
        noaa_ars TEXT,

        r_value REAL,
        totusjh REAL,
        totbsq REAL,
        totpot REAL,
        totusjz REAL,
        absnjzh REAL,
        savncpp REAL,
        usflux REAL,
        totfz REAL,
        meanpot REAL,
        epsx REAL,
        epsy REAL,
        epsz REAL,
        meanshr REAL,
        shrgt45 REAL,
        meangam REAL,
        meangbt REAL,
        meangbz REAL,
        meangbh REAL,
        meanjzh REAL,
        totfy REAL,
        meanjzd REAL,
        meanalp REAL,
        totfx REAL,

        source_system TEXT NOT NULL,
        source_series TEXT NOT NULL,
        fetched_at_utc TEXT NOT NULL,

        created_at_utc TEXT NOT NULL,

        UNIQUE(harpnum, t_rec)
    )
    """)

    conn.commit()


# -----------------------------
# 3) KOLON İSİMLERİNİ STANDARTLAŞTIRMA
# -----------------------------
def standardize_column_names(df):
    df = df.copy()

    rename_map = {
        "HARPNUM": "harpnum",
        "T_REC": "t_rec",
        "NOAA_ARS": "noaa_ars",
        "R_VALUE": "r_value",
        "TOTUSJH": "totusjh",
        "TOTBSQ": "totbsq",
        "TOTPOT": "totpot",
        "TOTUSJZ": "totusjz",
        "ABSNJZH": "absnjzh",
        "SAVNCPP": "savncpp",
        "USFLUX": "usflux",
        "TOTFZ": "totfz",
        "MEANPOT": "meanpot",
        "EPSX": "epsx",
        "EPSY": "epsy",
        "EPSZ": "epsz",
        "MEANSHR": "meanshr",
        "SHRGT45": "shrgt45",
        "MEANGAM": "meangam",
        "MEANGBT": "meangbt",
        "MEANGBZ": "meangbz",
        "MEANGBH": "meangbh",
        "MEANJZH": "meanjzh",
        "TOTFY": "totfy",
        "MEANJZD": "meanjzd",
        "MEANALP": "meanalp",
        "TOTFX": "totfx",
    }

    df = df.rename(columns=rename_map)
    return df


# -----------------------------
# 4) BARİZ TEKNİK HATALARI AYIKLAMA
# -----------------------------
def clean_raw_dataframe(df):
    df = df.copy()

    # Gerekli kolonlar var mı?
    required_cols = ["harpnum", "t_rec"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Gerekli kolon eksik: {col}")

    # harpnum sayıya çevrilir
    df["harpnum"] = pd.to_numeric(df["harpnum"], errors="coerce")

    # NOAA boş/missing olabilir, string olarak tut
    if "noaa_ars" in df.columns:
        df["noaa_ars"] = df["noaa_ars"].astype(str).replace({
            "nan": None,
            "None": None,
            "MISSING": None
        })

    # Sayısal feature'ları numerik yap
    numeric_cols = [
        "r_value", "totusjh", "totbsq", "totpot", "totusjz", "absnjzh",
        "savncpp", "usflux", "totfz", "meanpot", "epsx", "epsy", "epsz",
        "meanshr", "shrgt45", "meangam", "meangbt", "meangbz", "meangbh",
        "meanjzh", "totfy", "meanjzd", "meanalp", "totfx"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Temel teknik temizlik:
    # - harpnum boşsa satırı at
    # - t_rec boşsa satırı at
    df = df.dropna(subset=["harpnum", "t_rec"])

    # harpnum integer olsun
    df["harpnum"] = df["harpnum"].astype(int)

    # inf / -inf varsa null yap
    df = df.replace([float("inf"), float("-inf")], pd.NA)

    # Tamamen aynı kayıtlar gelmişse DataFrame içinde tekilleştir
    df = df.drop_duplicates(subset=["harpnum", "t_rec"], keep="last")

    return df


# -----------------------------
# 5) METADATA EKLEME
# -----------------------------
def add_metadata(df):
    df = df.copy()

    fetched_at_utc = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    created_at_utc = fetched_at_utc

    df["source_system"] = "JSOC"
    df["source_series"] = "hmi.sharp_720s_nrt"
    df["fetched_at_utc"] = fetched_at_utc
    df["created_at_utc"] = created_at_utc

    return df


# -----------------------------
# 6) SQLITE'A YAZMA
# -----------------------------
def save_to_sqlite(conn, df):
    if df.empty:
        print("Kaydedilecek veri yok.")
        return 0

    cursor = conn.cursor()

    insert_sql = f"""
    INSERT OR IGNORE INTO {TABLE_NAME} (
        harpnum, t_rec, noaa_ars,
        r_value, totusjh, totbsq, totpot, totusjz, absnjzh,
        savncpp, usflux, totfz, meanpot, epsx, epsy, epsz,
        meanshr, shrgt45, meangam, meangbt, meangbz, meangbh,
        meanjzh, totfy, meanjzd, meanalp, totfx,
        source_system, source_series, fetched_at_utc, created_at_utc
    )
    VALUES (
        ?, ?, ?,
        ?, ?, ?, ?, ?, ?,
        ?, ?, ?, ?, ?, ?, ?,
        ?, ?, ?, ?, ?, ?,
        ?, ?, ?, ?, ?,
        ?, ?, ?, ?
    )
    """

    rows_inserted = 0

    for _, row in df.iterrows():
        values = (
            row.get("harpnum"),
            row.get("t_rec"),
            row.get("noaa_ars"),

            row.get("r_value"),
            row.get("totusjh"),
            row.get("totbsq"),
            row.get("totpot"),
            row.get("totusjz"),
            row.get("absnjzh"),

            row.get("savncpp"),
            row.get("usflux"),
            row.get("totfz"),
            row.get("meanpot"),
            row.get("epsx"),
            row.get("epsy"),
            row.get("epsz"),

            row.get("meanshr"),
            row.get("shrgt45"),
            row.get("meangam"),
            row.get("meangbt"),
            row.get("meangbz"),
            row.get("meangbh"),

            row.get("meanjzh"),
            row.get("totfy"),
            row.get("meanjzd"),
            row.get("meanalp"),
            row.get("totfx"),

            row.get("source_system"),
            row.get("source_series"),
            row.get("fetched_at_utc"),
            row.get("created_at_utc"),
        )

        cursor.execute(insert_sql, values)

        if cursor.rowcount > 0:
            rows_inserted += 1

    conn.commit()
    return rows_inserted


# -----------------------------
# 7) TÜM AKIŞ
# -----------------------------
def run_once(hours=6):
    client = drms.Client()
    conn = sqlite3.connect(DB_PATH)

    try:
        create_table_if_not_exists(conn)

        raw_df = fetch_recent_regions(client, hours=hours)

        if raw_df.empty:
            print("Hiç veri gelmedi.")
            return

        print(f"Çekilen satır sayısı: {len(raw_df)}")

        df = standardize_column_names(raw_df)
        df = clean_raw_dataframe(df)
        df = add_metadata(df)

        inserted = save_to_sqlite(conn, df)

        print(f"Temizlik sonrası satır sayısı: {len(df)}")
        print(f"SQLite'a eklenen yeni kayıt sayısı: {inserted}")
        print(f"Veritabanı dosyası: {DB_PATH}")
        print(f"Tablo: {TABLE_NAME}")

    finally:
        conn.close()


if __name__ == "__main__":
    run_once(hours=6)

Sorgu: hmi.sharp_720s_nrt[][2026.03.27_17:40:14_TAI/6h@12m]
Çekilen satır sayısı: 267
Temizlik sonrası satır sayısı: 267
SQLite'a eklenen yeni kayıt sayısı: 28
Veritabanı dosyası: /Users/mustafaseyyitdogan/Desktop/detection_of_solar_flares/hackathon_detection_of_solar_flares/app/modules/data/fetch_data/raw_data/space_weather.db
Tablo: sharp_raw


## Kaydedilen Verinin İçeriğini Görme

In [28]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM sharp_raw ORDER BY id DESC LIMIT 10", conn)
conn.close()

print(df)

    id  harpnum                    t_rec noaa_ars  r_value   totusjh totbsq  \
0  532    13398  2026.03.27_22:24:00_TAI     None    0.000    20.574   None   
1  531    13398  2026.03.27_22:12:00_TAI     None    0.000    19.810   None   
2  530    13398  2026.03.27_22:00:00_TAI     None    0.000    18.891   None   
3  503    13395  2026.03.27_22:24:00_TAI     None    3.660   199.188   None   
4  502    13395  2026.03.27_22:12:00_TAI     None    3.615   194.381   None   
5  501    13395  2026.03.27_22:00:00_TAI     None    3.591   191.454   None   
6  480    13393  2026.03.27_22:24:00_TAI     None    3.690   710.183   None   
7  479    13393  2026.03.27_22:12:00_TAI     None    3.716   699.613   None   
8  478    13393  2026.03.27_22:00:00_TAI     None    3.716   701.726   None   
9  457    13392  2026.03.27_22:24:00_TAI     None    4.295  2222.068   None   

         totpot       totusjz  absnjzh  ...  meangbh   meanjzh totfy  \
0  1.325755e+21  5.920254e+11    2.156  ...   42.450 -0.00

## Veritabanındaki Veriyi CSV dosyasına çevirme

In [29]:
import sqlite3
import pandas as pd

# -----------------------------
# AYARLAR
# -----------------------------
CSV_PATH = "sharp_raw_export.csv"


def export_sqlite_to_csv(db_path, table_name, csv_path):
    # Veritabanına bağlan
    conn = sqlite3.connect(db_path)

    try:
        # Tabloyu DataFrame olarak çek
        query = f"SELECT * FROM {table_name}"
        df = pd.read_sql_query(query, conn)

        # CSV'ye yaz
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        print(f"CSV oluşturuldu: {csv_path}")
        print(f"Toplam satır: {len(df)}")

    except Exception as e:
        print("Hata:", e)

    finally:
        conn.close()


if __name__ == "__main__":
    export_sqlite_to_csv(DB_PATH, TABLE_NAME, CSV_PATH)

CSV oluşturuldu: sharp_raw_export.csv
Toplam satır: 293
